# YOLOv11s Detection — IRRG

Trains, validates, and tests a YOLOv11s object-detection model on Infrared-Red-Green 512×512 patches (50 % training overlap). Colour augmentation is disabled because IRRG composites encode spectral reflectance rather than perceptual colour.

**Install dependencies**

In [ ]:
# Install only once
!pip install ultralytics

**Import libraries**

In [ ]:
import json
import os
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Image
from ultralytics import YOLO

random.seed(42)

# Locate project root (walks upward until the 'src' package is found)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.paths import PROJECT_ROOT

**Define paths and run name**

In [ ]:
# Path to the YOLO data configuration file that lists the train/val/test
# image directories and class names
YAML_DIR = PROJECT_ROOT / "configs/yolov11s_object_detection_irrg_config_v01.yaml"

# Root directory where Ultralytics creates the run sub-folder.
# After training the structure will be:
#   <SAVE_DIR> / <RUN_NAME> / weights / best.pt
SAVE_DIR = PROJECT_ROOT / "models" / "yolov11_detection"

# Name for this specific run
RUN_NAME = "irrg"

**Load pre-trained YOLO weights**

In [ ]:
model = YOLO("yolo11s.pt")

**Training function**

In [ ]:
def yolo_train(
    model,
    data_yaml: str,
    epochs: int,
    imgsz: int,
    batch: int,
    name: str,
    project: str,
    device: int,
    workers: int,
    box: float,
    cls: float,
    hsv_h: float,
    hsv_s: float,
    hsv_v: float,
    auto_augment,
    erasing: float,
    mosaic: float,
    translate: float,
    max_retries: int = 5,
) -> None:
    """
    Train a YOLO detection model on IRRG imagery with an automatic retry
    mechanism and explicit colour-augmentation controls.

    IRRG composites encode spectral information rather than perceptual colour,
    so standard colour jitter (HSV shifts, AutoAugment) is disabled to prevent
    the model from learning spurious colour correlations that would not
    generalise across sensors or dates.  Mosaic and random-erasing are also
    disabled because the aerial patches already provide sufficient spatial
    diversity.

    Args:
        model:        Loaded YOLO model instance.
        data_yaml:    Path to the Ultralytics data config (.yaml).
        epochs:       Total training epochs.
        imgsz:        Input image size (square, in pixels).
        batch:        Batch size.
        name:         Sub-folder name for this run inside 'project'.
        project:      Root output directory (SAVE_DIR).
        device:       GPU index or 'cpu'.
        workers:      DataLoader worker processes.
        box:          Box-loss weight.
        cls:          Class-loss weight.
        hsv_h:        Hue jitter fraction (0.0 = disabled).
        hsv_s:        Saturation jitter fraction (0.0 = disabled).
        hsv_v:        Value jitter fraction (0.0 = disabled).
        auto_augment: AutoAugment policy name, or None to disable.
        erasing:      Random-erasing probability (0.0 = disabled).
        mosaic:       Mosaic augmentation probability (0.0 = disabled).
        translate:    Translation fraction (0.0 = disabled).
        max_retries:  Maximum recovery attempts on OSError.
    """
    def _log_gpu(prefix: str = "") -> None:
        if torch.cuda.is_available():
            mb = torch.cuda.memory_allocated(device) / (1024 ** 2)
            print(f"{prefix} GPU memory allocated: {mb:.2f} MB")

    _log_gpu("[Before training]")

    for attempt in range(max_retries):
        try:
            start = time.time()
            model.train(
                data=data_yaml,
                epochs=epochs,
                imgsz=imgsz,
                batch=batch,
                name=name,
                project=project,
                device=device,
                workers=workers,
                box=box,
                cls=cls,
                hsv_h=hsv_h,
                hsv_s=hsv_s,
                hsv_v=hsv_v,
                auto_augment=auto_augment,
                erasing=erasing,
                mosaic=mosaic,
                translate=translate,
            )
            elapsed = time.time() - start
            print(f"\nTraining complete in {elapsed:.0f} s")
            _log_gpu("[After training]")
            return

        except OSError as exc:
            print(f"OSError on attempt {attempt + 1}/{max_retries}: {exc}")
            if attempt < max_retries - 1:
                wait = 2 ** (attempt + 1)
                print(f"Retrying in {wait} s …")
                time.sleep(wait)
            else:
                print("Max retries reached — training aborted.")

        except Exception as exc:
            print(f"Unexpected error: {exc}")
            break

**Run training**

In [ ]:
yolo_train(
    model,
    data_yaml    = str(YAML_DIR),
    epochs       = 100,
    imgsz        = 640,
    batch        = 16,
    name         = RUN_NAME,
    project      = str(SAVE_DIR),
    device       = 0,
    workers      = 0,
    box          = 12.0,
    cls          = 2.0,
    # Colour augmentation disabled: IRRG bands encode spectral reflectance,
    # not perceptual colour, so HSV jitter would corrupt the signal.
    hsv_h        = 0.0,
    hsv_s        = 0.0,
    hsv_v        = 0.0,
    auto_augment = None,
    erasing      = 0.0,
    mosaic       = 0.0,
    translate    = 0.0,
)

**Load best checkpoint**

In [ ]:
# The best checkpoint is always saved at <SAVE_DIR>/<RUN_NAME>/weights/best.pt.
# Using RUN_DIR means you only need to change SAVE_DIR / RUN_NAME at the top
# of the notebook — no path duplication here.
model = YOLO(SAVE_DIR / RUN_NAME / "weights" / "best.pt")

**Validate on val split**

In [ ]:
results = model.val(
    data=str(YAML_DIR),
    split="val",
    plots=True,
    save_json=True,
)

# Extract per-class precision, recall, mAP50, mAP50-95
class_metrics = []
for i, name in results.names.items():
    precision, recall, map50, map5095 = results.class_result(i)
    class_metrics.append([name, precision, recall, map50, map5095])

df_val = pd.DataFrame(
    class_metrics,
    columns=["Class", "Precision", "Recall", "mAP50", "mAP50-95"],
)
print("Validation metrics:\n")
print(df_val)

out_csv = SAVE_DIR / RUN_NAME / "val_metrics.csv"
df_val.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

**Plot training loss curves**

In [ ]:
results_csv = SAVE_DIR / RUN_NAME / "results.csv"
df = pd.read_csv(results_csv, on_bad_lines="skip")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df["epoch"], df["train/box_loss"], label="Train Box Loss")
ax.plot(df["epoch"], df["val/box_loss"],   label="Val Box Loss",   linestyle="--")
ax.plot(df["epoch"], df["train/cls_loss"], label="Train Cls Loss")
ax.plot(df["epoch"], df["val/cls_loss"],   label="Val Cls Loss",   linestyle="--")
ax.plot(df["epoch"], df["train/dfl_loss"], label="Train DFL Loss")
ax.plot(df["epoch"], df["val/dfl_loss"],   label="Val DFL Loss",   linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training vs Validation Loss")
ax.legend()
ax.grid(True)
fig.tight_layout()
fig.savefig(SAVE_DIR / RUN_NAME / "loss_curves.png", dpi=150)
plt.show()

**Evaluate on test split**

In [ ]:
results = model.val(
    data=str(YAML_DIR),
    split="test",
    plots=True,
    save_json=True,
)

class_metrics = []
for i, name in results.names.items():
    precision, recall, map50, map5095 = results.class_result(i)
    class_metrics.append([name, precision, recall, map50, map5095])

df_test = pd.DataFrame(
    class_metrics,
    columns=["Class", "Precision", "Recall", "mAP50", "mAP50-95"],
)
print("Test metrics:\n")
print(df_test)

out_csv = SAVE_DIR / RUN_NAME / "test_metrics.csv"
df_test.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

**Per-class performance bar chart (test set)**

In [ ]:
df = pd.read_csv(SAVE_DIR / RUN_NAME / "test_metrics.csv")
metrics   = ["Precision", "Recall", "mAP50", "mAP50-95"]
n_classes = len(df)
bar_width = 0.18
x         = np.arange(n_classes)

fig, ax = plt.subplots(figsize=(14, 7))
for i, metric in enumerate(metrics):
    ax.bar(x + i * bar_width, df[metric], width=bar_width, label=metric)

ax.set_xticks(x + bar_width * (len(metrics) - 1) / 2)
ax.set_xticklabels(df["Class"], rotation=45, ha="right")
ax.set_ylabel("Metric Value")
ax.set_title("Per-Class Performance — Test Set")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.6)
fig.tight_layout()
fig.savefig(SAVE_DIR / RUN_NAME / "test_metrics_per_class.png", dpi=300)
plt.show()